### Object Tracking
#### Objective:
After implementing template matching, students should explore potential improvements.  The key challenges with template matching are: 1. Scale changes 2. Rotation Changes 3. Brightness/ contrast changes and Occlusions.
#### Implementation Details:
Template matching relies on raw pixel values, making it sensitive to changes. Instead of comparing pixel intensities, extract SIFT features and follow the algorithm for tracking by feature detection discussed in the lecture. You can use cv2.SIFT_create(), sift.detectAndCompute,cv2.BFMatcher(), cv2.drawMatches() function here to compare. 

<div class="alert alert-block alert-info">
 As this was a graded coursework, some of the contents have been redacted to prevent plagiarism. Full version is available upon request.
    </div>

In [ ]:
import cv2
import numpy as np
import warnings

class ImprovedTemplateMatcher:
    def __init__(self):
        pass
    
    def getSelectionMask(self, frame, selection):
        """
        Generates a mask which covers all the pixels inside the selection box
        """
        ### REDACTED ###
    
    def getBackgroundMask(self, frame, selection):
        """
        Generates a mask which covers all the pixels outside the selection box
        """
        ### REDACTED ###
    
    def findTemplateFeatures(self, frame, selection, n = 50):
        """
        Uses the first frame to find all the key points in an image.

        Classify them as either background or object based on the whether they are within the selection area or not.
        """
        #Generate two masks, one for the selection and another for the background
        selectionMask = self.getSelectionMask(frame, selection)
        backgroundMask = self.getBackgroundMask(frame, selection)

        sift = cv2.SIFT_create()
        
        #Detect Object and Background features separately
        ### REDACTED ###

        if kpObj is None or desObj is None or kpBg is None or desBg is None or len(kpObj) < 2 or len(kpBg) < 2:
            print("Warning: Not enough keypoints detected in template.")
            return
        
        ### REDACTED ###

        self.previousBox = selection

    def REDACTED_FUNC():
        ### REDACTED ###

    def getBoundingBox(self, keyPoints, matches):
        """
        REDACTED
        """
        ### REDACTED ###

    def REDACTED_FUNC():
        ### REDACTED ###

    def REDACTED_FUNC():
        ### REDACTED ###
    
    def REDACTED_FUNC():
        ### REDACTED ###
    
    def getScore(self, previousBox,currentBox, keyPoints, matches):
        prevX, prevY, prevW, prevH = previousBox
        currX, currY, currW, currH = currentBox  

        #The amount of change between the current boundingBox and the previous one
        heuristic = ### REDACTED ###

        #For all keypoints inside the box, if it is an object, add +2. If it is background, -1
        confidenceScore = 0
        
        #For all points inside the bounding box, remove 1 score
        for kp in keyPoints:
            
            #if kp.pt is within currentBox
            if kp.pt[0] >= currX and kp.pt[0] < currX + currW and kp.pt[1] >= currY and kp.pt[1] < currY + currH:
                confidenceScore -= 2

        #For all matches inside the bounding box add 3 so that -1 + 3 = 2
        for m in matches:
            if keyPoints[m.queryIdx].pt[0] >= currX and keyPoints[m.queryIdx].pt[0] < currX + currW and keyPoints[m.queryIdx].pt[1] >= currY and keyPoints[m.queryIdx].pt[1] < currY + currH:
                confidenceScore += 3

        #Calculate the Match Score
        return confidenceScore - heuristic

    def feature_based_matching(self, frame, ratio_thresh = 0.7, min_matches = 5, ransac_thresh = 5, n = 50):      
        sift = cv2.SIFT_create()
        bfm = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
        #Detect the features from the frame
        kp, des = sift.detectAndCompute(frame, None)
        
        #Get the features from the previous model
        kpObj, desObj = self.objectModel
        kpBg, desBg = self.backgroundModel

        objectMatches = bfm.match(des, desObj)
        backgroundMatches = bfm.match(des, desBg)

        matchesLen = np.array(objectMatches).shape[0]

        close_matches = []
        ### REDACTED ###
        
        #If the model did not find enough matches, skip the frame
        if len(close_matches) < min_matches:
            print("NOT ENOUGH MATCHES FOUND")
            return

        currentBox = self.previousBox
        bestBox = self.previousBox
        highestScore = -np.inf

        subset = np.copy(close_matches)
        #Try fitting the bounding box around all matches, then remove the furthest one from the mean
        ### REDACTED ###

        self.originalBox = bestBox

        #Interpolate between the previous bounding box and the current one
        bestBox = ### REDACTED ###
        #Limit the movement to prevent it from moving too much
        self.previousBox = ### REDACTED ###

        x, y, w, h = self.previousBox
        #Update the object model with all the key points inside the bounding box
        ### REDACTED ###

        x, y, w, h = self.originalBox
        self.adjustedBox = ### REDACTED ###

        x, y, w, h = ### REDACTED ###
        self.meanBox = ### REDACTED ###

        return
    
        
    def draw_bounding_box(self, frame):
        """
        Draw bounding box around the detected object.
        """
        frame_copy = frame.copy()
        x, y, w, h = self.previousBox
        pts = np.float32([ [x, y],[x, y+h],[x+w,y+h],[x+w, y] ]).reshape(-1,1,2)
        cv2.polylines(frame_copy, [np.int32(pts)], True, (0,255,0), 1, cv2.LINE_AA)
        x, y, w, h = self.adjustedBox
        pts = np.float32([ [x, y],[x, y+h],[x+w,y+h],[x+w, y] ]).reshape(-1,1,2)
        cv2.polylines(frame_copy, [np.int32(pts)], True, (255,0,0), 1, cv2.LINE_AA)
        x, y, w, h = self.originalBox
        pts = np.float32([ [x, y],[x, y+h],[x+w,y+h],[x+w, y] ]).reshape(-1,1,2)
        cv2.polylines(frame_copy, [np.int32(pts)], True, (0,0,255), 1, cv2.LINE_AA)
        x, y, w, h = self.meanBox
        pts = np.float32([ [x, y],[x, y+h],[x+w,y+h],[x+w, y] ]).reshape(-1,1,2)
        cv2.polylines(frame_copy, [np.int32(pts)], True, (0,255,255), 1, cv2.LINE_AA)
        return frame_copy

def runObjectTracking(input_video_path, min_matches, ransac_threshold, lowe_threshold, sift_n):
    cap = cv2.VideoCapture(input_video_path)
    _, first_frame = cap.read()
    if first_frame is None:
        warnings.warn("Video file not found. Please select a path to a valid video")
        return

    selection = cv2.selectROI("Select Template", first_frame, fromCenter=False, showCrosshair=True)
    x, y, w, h = selection
    cv2.destroyWindow("Select Template")

    if w > 0 and h > 0:
        template = first_frame[y:y+h, x:x+w]
    else:
        print("Error: No template selected or invalid ROI.")
        cap.release()
        exit()

    matcher = ImprovedTemplateMatcher()
    frame_size = first_frame.shape[:2]
    template_size = template.shape[:2]

    cv2.namedWindow('Tracking Result', cv2.WINDOW_NORMAL)
    cv2.resizeWindow('Tracking Result', frame_size[0], frame_size[1])
    cv2.namedWindow('Matches Visualization', cv2.WINDOW_NORMAL)
    cv2.resizeWindow('Matches Visualization', frame_size[0] + template_size[0], frame_size[1])

    matcher.findTemplateFeatures(first_frame, selection, sift_n)

    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            print("End of video or error reading frame.")
            break

        frame_count += 1
        print(f"\n--- Processing Frame {frame_count} ---")

        matcher.feature_based_matching(frame, lowe_threshold, min_matches, ransac_threshold, sift_n)
        frame_with_box = matcher.draw_bounding_box(frame)
        cv2.imshow('Tracking Result', frame_with_box)

        if cv2.waitKey(60) & 0xFF == ord('q'):
            print("Exit requested by user.")
            break
        
            
    cap.release()
    cv2.destroyAllWindows()


#### Runing the code

Run the cell above to load the code

Run the cell below to execute it

This will open a new window displaying the first frame of the video

Select the object to be tracked by dragging a bounding box around the object (the girl's face when using the sample video) using your mouse. Make sure to be generous when making the selection by covering the immediate area around the object too

Keep trying until you are satisfied with your selection

Press the 'Enter' key to confirm your selection

Multiple colour-coded tracking algorithms will be displayed. This can be used to assess their performance

You can press the 'q' key to exit at any time

In [ ]:
# The video to be loaded
input_video_path = 'testFiles\ObjectTracking_video.mp4'  # Replace with your video file

min_matches = 3
ransac_threshold = 7
lowe_threshold = 0.75
sift_n = 200

runObjectTracking(input_video_path, min_matches, ransac_threshold, lowe_threshold, sift_n)